In [1]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.integrate as integrate
from scipy import optimize
import plotly.graph_objects as go
import plotly.io as pio

In [ ]:
eta_0 = 120 * np.pi  # Free space impedance
class layer:
    def __init__(self, f, eps, mu, theta, polarization,l=0, l_in_parts_of_wavelength=False, array_mode=False):
        self.eps = eps
        self.mu = mu
        self.l = l
        self.polarization = polarization
        self.theta = theta
        self.f = f
        self.array_mode = array_mode
        self.w = 2 * np.pi * f # [:,None, None]
        mu0 = 4e-7 * np.pi
        eps0 = 8.854e-12
        eta_0 = np.sqrt(mu0/eps0)
        c = 1/np.sqrt(mu0*eps0)/1e9
        self.n = np.sqrt(mu * eps)
        if polarization == 'vert':
            self.z = eta_0/self.eps * np.sqrt(self.n**2 - (np.sin(theta)**2)) #[None, None, :]
        elif polarization == 'horiz':
            self.z = eta_0 * self.mu/np.sqrt(self.n**2 - (np.sin(theta)**2)) #[None, None, :]
        self.beta = self.w/c * np.sqrt(self.n**2 - (np.sin(theta)**2)) #[:, None, :]
        if l_in_parts_of_wavelength:
            self.l = l * 2*np.pi/self.beta
        self.phi = self.beta * self.l #[:, None, :]
    def abcd(self):
        abcd_res = np.array([[np.cos(self.phi), -1j*self.z*np.sin(self.phi)],
                        [-1j*(1/self.z)*np.sin(self.phi), np.cos(self.phi)]], dtype=complex)
        if self.array_mode:
            return abcd_res.transpose(2,3,4,0,1)
        return abcd_res